# MobileNetV2 autoencoder on CIFAR-10

A convolutional autoencoder built from MobileNetV2 inverted residual blocks. The encoder turns a 32x32 RGB image into a 320-dimensional code (320 channels at 1x1). The decoder mirrors it with transposed convolutions and ends with a tanh. The model is trained with an MSE loss on the CIFAR-10 training set and evaluated on the 10,000 test images.

A block adds its input to its output when its stride is 1 and its input and output channels match, as in MobileNetV2.

On Colab, pick a GPU runtime (Runtime > Change runtime type) before running all cells.

In [ ]:
import torch
import torch.nn as nn

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, expansion: int, stride: int) -> None:
        super(InvertedResidualBlock, self).__init__()

        self.use_residual = stride == 1 and in_channels == out_channels
        self.expansion_layer = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * expansion, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels * expansion),
            nn.ReLU6(inplace=True)
        )

        self.depthwise_layer = nn.Sequential(
            nn.Conv2d(in_channels * expansion, in_channels * expansion, kernel_size=3, stride=stride, padding=1, groups=in_channels * expansion, bias=False),
            nn.BatchNorm2d(in_channels * expansion),
            nn.ReLU6(inplace=True)
        )

        self.pointwise_layer = nn.Sequential(
            nn.Conv2d(in_channels * expansion, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.expansion_layer(x)
        out = self.depthwise_layer(out)
        out = self.pointwise_layer(out)
        if self.use_residual:
            out = out + x
        return out

class InvertedResidualBlockTranspose(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, t: int = 6, stride: int = 1) -> None:
        super(InvertedResidualBlockTranspose, self).__init__()

        self.use_residual = stride == 1 and in_channels == out_channels

        self.expansion_layer = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * t, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels * t),
            nn.ReLU6(inplace=True)
        )

        self.depthwise_layer_transpose = nn.Sequential(
            nn.ConvTranspose2d(in_channels * t, in_channels * t, kernel_size=3, stride=stride, padding=1,
                               output_padding=(0 if stride == 1 else 1), groups=in_channels * t, bias=False),
            nn.BatchNorm2d(in_channels * t),
            nn.ReLU6(inplace=True)
        )

        self.pointwise_layer_transpose = nn.Sequential(
            nn.ConvTranspose2d(in_channels * t, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.expansion_layer(x)
        out = self.depthwise_layer_transpose(out)
        out = self.pointwise_layer_transpose(out)
        if self.use_residual:
            out = out + x
        return out


class MobileNetV2CAE(nn.Module):
    def __init__(self, in_channels: int = 3) -> None:
        super(MobileNetV2CAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            InvertedResidualBlock(32, 16, 1, 1),
            InvertedResidualBlock(16, 24, 6, 2),
            InvertedResidualBlock(24, 24, 6, 1),
            InvertedResidualBlock(24, 32, 6, 2),
            InvertedResidualBlock(32, 32, 6, 1),
            InvertedResidualBlock(32, 32, 6, 1),
            InvertedResidualBlock(32, 64, 6, 2),
            InvertedResidualBlock(64, 64, 6, 1),
            InvertedResidualBlock(64, 64, 6, 1),
            InvertedResidualBlock(64, 64, 6, 1),
            InvertedResidualBlock(64, 96, 6, 1),
            InvertedResidualBlock(96, 96, 6, 1),
            InvertedResidualBlock(96, 96, 6, 1),
            InvertedResidualBlock(96, 160, 6, 2),
            InvertedResidualBlock(160, 160, 6, 1),
            InvertedResidualBlock(160, 160, 6, 1),
            InvertedResidualBlock(160, 320, 6, 1)
        )

        self.decoder = nn.Sequential(
            InvertedResidualBlockTranspose(320, 160, 6, 1),
            InvertedResidualBlockTranspose(160, 160, 6, 1),
            InvertedResidualBlockTranspose(160, 160, 6, 2),
            InvertedResidualBlockTranspose(160, 96, 6, 1),
            InvertedResidualBlockTranspose(96, 96, 6, 1),
            InvertedResidualBlockTranspose(96, 96, 6, 1),
            InvertedResidualBlockTranspose(96, 64, 6, 1),
            InvertedResidualBlockTranspose(64, 64, 6, 1),
            InvertedResidualBlockTranspose(64, 64, 6, 1),
            InvertedResidualBlockTranspose(64, 64, 6, 1),
            InvertedResidualBlockTranspose(64, 32, 6, 2),
            InvertedResidualBlockTranspose(32, 32, 6, 1),
            InvertedResidualBlockTranspose(32, 32, 6, 1),
            InvertedResidualBlockTranspose(32, 24, 6, 2),
            InvertedResidualBlockTranspose(24, 24, 6, 1),
            InvertedResidualBlockTranspose(24, 16, 6, 2),
            nn.ConvTranspose2d(16, in_channels, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.encoder(x)
        x = self.decoder(x)
        return x


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV2CAE(in_channels=3).to(device).eval()
with torch.no_grad():
    latent = model.encoder(torch.zeros(2, 3, 32, 32, device=device))
print(f'Device: {torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu"}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Latent code of one image: {tuple(latent.shape[1:])}')
print(f'Blocks with a skip connection: {sum(1 for m in model.modules() if getattr(m, "use_residual", False))}')


## Data

CIFAR-10 through torchvision. Pixels are scaled to [-1, 1] to match the tanh output, so the MSE values below are on that scale; divide by 4 for the MSE on [0, 1] pixels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import torchvision.transforms as transforms
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

train_dataset = CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)
print(f'{len(train_dataset)} training images, {len(test_dataset)} test images')

## Training

25 epochs of Adam (learning rate 0.001, batch size 100) on the MSE between each image and its reconstruction, the same recipe as the 2023 run. After every epoch the model is switched to eval mode and scored on the full test set.

In [ ]:
num_epochs = 25
learning_rate = 0.001

torch.manual_seed(0)
model = MobileNetV2CAE(in_channels=3).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


def test_mse(autoencoder: nn.Module, loader: DataLoader[tuple[torch.Tensor, int]]) -> float:
    autoencoder.eval()
    squared_error = 0.0
    value_count = 0
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            squared_error += nn.functional.mse_loss(autoencoder(images), images, reduction='sum').item()
            value_count += images.numel()
    return squared_error / value_count

In [ ]:
train_history: list[float] = []
test_history: list[float] = []

for epoch in range(num_epochs):
    model.train()
    loss_sum = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        outputs = model(images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * images.size(0)

    train_history.append(loss_sum / len(train_dataset))
    test_history.append(test_mse(model, test_loader))
    print(f'Epoch [{epoch + 1}/{num_epochs}]  train MSE {train_history[-1]:.4f}  test MSE {test_history[-1]:.4f}')

## Results

Test MSE of the final model, then the train and test MSE across epochs.

In [ ]:
figure_dir = Path('figures')
figure_dir.mkdir(exist_ok=True)

final_test_mse = test_history[-1]
print(f'Test MSE, [-1, 1] scale: {final_test_mse:.4f}')
print(f'Test MSE, [0, 1] scale: {final_test_mse / 4:.4f}')
print(f'PSNR from the mean test MSE: {10 * np.log10(4 / final_test_mse):.2f} dB')

epochs = np.arange(1, num_epochs + 1)
fig, axis = plt.subplots(figsize=(7, 4))
axis.plot(epochs, train_history, color='#2a78d6', linewidth=2, label='train (mean over the epoch)')
axis.plot(epochs, test_history, color='#eb6834', linewidth=2, label='test (end of the epoch)')
axis.set_xlabel('Epoch')
axis.set_ylabel('MSE, [-1, 1] scale')
axis.set_title('Reconstruction error on CIFAR-10')
axis.grid(color='#e1e0d9', linewidth=0.8)
axis.spines[['top', 'right']].set_visible(False)
axis.legend(frameon=False)
fig.savefig(figure_dir / 'loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

The first test image of each class (top) and its reconstruction (bottom).

In [ ]:
class_names: list[str] = test_dataset.classes
first_of_class = [test_dataset.targets.index(label) for label in range(len(class_names))]
inputs = torch.stack([test_dataset[index][0] for index in first_of_class])

model.eval()
with torch.no_grad():
    reconstructions = model(inputs.to(device)).cpu()

fig, axes = plt.subplots(2, len(class_names), figsize=(15, 3.4))
for column, class_name in enumerate(class_names):
    for row, batch in enumerate((inputs, reconstructions)):
        axes[row, column].imshow((batch[column] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
    axes[0, column].set_title(class_name)
axes[0, 0].set_ylabel('input')
axes[1, 0].set_ylabel('reconstruction')
fig.savefig(figure_dir / 'reconstructions.png', dpi=120, bbox_inches='tight')
plt.show()

## Latent space

t-SNE of the 320-dimensional codes of all 10,000 test images. Each panel highlights one class over the others. The labels play no part in training.

In [ ]:
codes: list[npt.NDArray[np.float32]] = []
labels: list[npt.NDArray[np.int64]] = []

model.eval()
with torch.no_grad():
    for images, targets in test_loader:
        codes.append(model.encoder(images.to(device)).flatten(1).cpu().numpy())
        labels.append(targets.numpy())

latent_space = np.concatenate(codes)
label_array = np.concatenate(labels)
latent_2d = TSNE(n_components=2, random_state=42).fit_transform(latent_space)

fig, axes = plt.subplots(2, 5, figsize=(13, 6), sharex=True, sharey=True)
for label, (axis, class_name) in enumerate(zip(axes.flat, class_names)):
    selected = label_array == label
    axis.scatter(latent_2d[~selected, 0], latent_2d[~selected, 1], s=2, color='#dddcd5', linewidths=0)
    axis.scatter(latent_2d[selected, 0], latent_2d[selected, 1], s=3, color='#2a78d6', linewidths=0)
    axis.set_title(class_name)
    axis.set_xticks([])
    axis.set_yticks([])
    for spine in axis.spines.values():
        spine.set_color('#e1e0d9')
fig.suptitle('t-SNE of the latent codes of the 10,000 test images, one class highlighted per panel')
fig.tight_layout()
fig.savefig(figure_dir / 'tsne_latent.png', dpi=110, bbox_inches='tight')
plt.show()
